# 02 — Population curve vs I_ML: which one goes into the live system?

Both methods get **the same input** (the causal variant-A angle and the same frames),
and are tested on **people they never saw**.

| Experiment | Train | Test | Question it answers |
|---|---|---|---|
| **E1** | Fit3D, 7 people | the 8th (×8) | does it work for a new person? |
| **E2a** | all REHAB24-6 | all Fit3D | does it survive a new room/camera/dataset? |
| **E2b** | all Fit3D | all REHAB24-6 | same, other direction |
| **E3** | both, 17 people | the 18th (×18) | final estimate with all data |

Methods: `raw` (variant A, no correction) · `curve` (one monotonic curve per view class,
the deployed D_2D_PerView) · `iml` (LightGBM 200 × 31 on the 16 per-frame features) ·
`iml_nolook` (the same without `std_angle` and `d_angle`, so it needs **no look-ahead** live).

Scores are **per recording** MAE (then median / IQR), and paired tests are **per person**,
so long recordings and frame autocorrelation do not inflate anything.

### Nested cross-validation (hyperparameters tuned without touching the test person)

With `TUNE = True`, every outer fold tunes its own hyperparameters **on its training people only**:

```
outer fold:  train people --> inner GroupKFold (4 folds, grouped by person)
                              try every setting in the grid, score it on the inner held-out people
                              pick the best --> refit on all train people --> predict the test person
```

So the test person (E1, E3) or the test dataset (E2) never influences a hyperparameter.
This removes the optimistic bias of using settings chosen earlier on REHAB24-6.
`curve` tunes `bin_width` and `min_count`; `iml` / `iml_nolook` tune trees, leaves,
learning rate and `min_child_samples`. The chosen setting per fold is logged, and how
stable it is across folds is itself a result. The feature set (the 4 per-recording
constants dropped) is a fixed design rule, not tuned.

In [ ]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from scipy.stats import wilcoxon
import lightgbm as lgb

REPO   = r'C:\path\to\AI-based-Physiotherapy-project-'   # <-- same as notebook 01 (ADA: os.path.expanduser('~/...'))
FILES  = {'rehab24': os.path.join(REPO, 'data', 'fit3d', 'feat_rehab24.npz'),
          'fit3d':   os.path.join(REPO, 'data', 'fit3d', 'feat_fit3d.npz')}
USE_GPU = True        # tried, falls back to CPU if LightGBM has no GPU build
def find_repo(path):
    '''use the path given; if it is wrong, look upwards from the notebook folder'''
    cands = [path, os.getcwd(), os.path.dirname(os.getcwd())]
    for c in cands:
        if os.path.exists(os.path.join(c, 'calibration.py')) and os.path.exists(os.path.join(c, 'pose', 'landmarks.py')):
            if c != path:
                print('REPO path not found as typed; using', c)
            return os.path.abspath(c)
    raise FileNotFoundError(f'repo not found at {path!r} - set REPO to the folder that holds calibration.py')
REPO = find_repo(REPO)
FILES = {k: os.path.join(REPO, 'data', 'fit3d', os.path.basename(v)) for k, v in FILES.items()}
for v in FILES.values():
    assert os.path.exists(v), f'missing {v}'
sys.path[:0] = [REPO, os.path.join(REPO, 'scripts')]
from calibration import fit_curve, apply_calibration, save_params
from fit_calibration import extend_below

DROP = {'bone_upper', 'bone_fore', 'sw_frac', 'is_right'}   # per-recording constants (benchmark Check A)

def view_class(v):
    '''CAMERA-relative view.
    REHAB24-6 labels describe the subject's orientation to camera 17. Camera 18 stands
    ~90 deg away, so "Profile/cam18" is seen nearly head-on and "Front/cam18" side-on
    (checked below with shoulder width in the image, sw_frac). Fit3D labels are already
    camera-relative (from the mocap shoulder line).'''
    v = str(v).lower()
    cam18 = 'cam18' in v
    if 'half' in v: return 'half'
    if 'front' in v: return 'profile' if cam18 else 'front'
    if 'prof' in v or 'side' in v: return 'front' if cam18 else 'profile'
    return 'unknown'

frames = []
for name, path in FILES.items():
    d = np.load(path, allow_pickle=False)
    if not bool(d['causal']):
        print(f'WARNING: {name} features were made with the non-causal filter')
    names = [str(n) for n in d['feature_names']]
    df = pd.DataFrame(d['X'], columns=names)
    for k in ('y', 'person', 'recording', 'view', 'exercise', 'dataset'):
        df[k] = d[k]
    df['vclass'] = df['view'].map(view_class)
    frames.append(df)
D = pd.concat(frames, ignore_index=True)
KEEP = [n for n in names if n not in DROP]
D = D[np.isfinite(D['angle_2d']) & np.isfinite(D['y'])].reset_index(drop=True)
print(D.groupby(['dataset', 'vclass']).agg(frames=('y', 'size'), people=('person', 'nunique'),
      recordings=('recording', 'nunique'), truth_p5=('y', lambda s: s.quantile(.05)),
      truth_p95=('y', lambda s: s.quantile(.95))).round(1))
print('features used by I_ML:', len(KEEP), KEEP)

# check of the camera-relative labels: shoulder width in the image is small side-on, large head-on
print(D.groupby(['dataset', 'view', 'vclass'])[['sw_frac', 'torso_ratio']].median().round(3))
if not (D.vclass[D.dataset == 'fit3d'] == 'profile').any():
    print('\nNOTE: Fit3D has no side-on (profile) camera. Profile-view results come from REHAB24-6 only;'
          '\n      E1 and E2a have no profile rows, E2b tests profile with NO profile training data.')

## Methods — each is fitted on the training rows only

In [ ]:
from sklearn.model_selection import GroupKFold

TUNE        = True   # nested CV; False = fixed settings below (quick run, ~10x faster)
INNER_FOLDS = 4
INNER_THIN  = 3      # inner loop uses every 3rd frame: neighbouring frames are near-duplicates

BASE = dict(objective='l1', min_child_samples=80, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, reg_lambda=1.0, verbosity=-1, n_jobs=-1,
            n_estimators=200, num_leaves=31, learning_rate=0.09)     # fixed default (benchmark 6b, causal)
CURVE_DEFAULT = dict(bin_width=10, min_count=300)

# grids: more trees need a smaller step, so trees / leaves / rate move together
GRID_IML = [dict(n_estimators=n, num_leaves=l, learning_rate=r, min_child_samples=m)
            for n, l, r in ((120, 15, 0.14), (200, 31, 0.09), (300, 31, 0.07), (600, 63, 0.05))
            for m in (40, 160)]
GRID_CURVE = [dict(bin_width=b, min_count=m) for b in (5, 10, 15) for m in (100, 300, 1000)]
TUNING_LOG, FOLD = [], {'label': ''}

def gpu_ok():
    try:
        lgb.LGBMRegressor(device_type='gpu', n_estimators=2, verbosity=-1).fit(
            np.random.rand(500, 3), np.random.rand(500))
        return True
    except Exception as e:
        print('LightGBM GPU not available, using CPU:', str(e).splitlines()[0][:100])
        return False
if USE_GPU and gpu_ok():
    BASE['device_type'] = 'gpu'
print('LightGBM device:', BASE.get('device_type', 'cpu'))

def fit_one_curve(raw, truth, bin_width=10, min_count=300):
    for mc in (min_count, 100, 30):          # thin data: relax min_count rather than fail
        try:
            return extend_below(fit_curve(raw, truth, bin_width=bin_width, min_count=mc))
        except ValueError:
            pass
    return None

def curve_predict(tr, te, bin_width=10, min_count=300):
    '''one curve per view class; a class with too little training data uses the pooled curve'''
    pooled = fit_one_curve(tr['angle_2d'], tr['y'], bin_width, min_count)
    out = np.empty(len(te))
    for vc, idx in te.groupby('vclass').indices.items():
        sub = tr[tr['vclass'] == vc]
        p = fit_one_curve(sub['angle_2d'], sub['y'], bin_width, min_count) if len(sub) >= 2000 else None
        out[idx] = apply_calibration(te['angle_2d'].to_numpy(float)[idx], p or pooled)
    return out

def iml_predict(tr, te, cols, **hp):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model = lgb.LGBMRegressor(**{**BASE, **hp}).fit(tr[cols], tr['y'])
        return np.clip(model.predict(te[cols]), 0, 180)

def rec_median_mae(df, pred):
    '''the reported metric: median over recordings of the per-recording MAE'''
    e = pd.Series(np.abs(pred - df['y'].to_numpy()))
    return float(e.groupby(df['recording'].to_numpy()).mean().median())

def tune(tr, predict, grid, name):
    '''inner grouped CV on the TRAINING people only -> best setting'''
    tr = tr.iloc[::INNER_THIN].reset_index(drop=True)
    k = min(INNER_FOLDS, tr['person'].nunique())
    folds = list(GroupKFold(n_splits=k).split(tr, groups=tr['person']))
    scores = []
    for hp in grid:
        s = [rec_median_mae(tr.iloc[va].reset_index(drop=True),
                            predict(tr.iloc[fi], tr.iloc[va].reset_index(drop=True), **hp))
             for fi, va in folds]
        scores.append(float(np.mean(s)))
    best = int(np.argmin(scores))
    TUNING_LOG.append(dict(fold=FOLD['label'], method=name, inner_mae=round(scores[best], 2), **grid[best]))
    return grid[best]

def m_raw(tr, te):
    return te['angle_2d'].to_numpy(float)

def m_curve(tr, te):
    hp = tune(tr, curve_predict, GRID_CURVE, 'curve') if TUNE else CURVE_DEFAULT
    return curve_predict(tr, te, **hp)

def _iml(cols, name):
    def fn(tr, te):
        pred = lambda a, b, **hp: iml_predict(a, b, cols, **hp)
        hp = tune(tr, pred, GRID_IML, name) if TUNE else {}
        return iml_predict(tr, te, cols, **hp)
    return fn

# std_angle (centred 15-frame window) is what forces the 7-frame look-ahead live;
# d_angle is dropped with it. Together they carry ~3 % of the gain in the benchmark.
KEEP_NOLOOK = [c for c in KEEP if c not in ('std_angle', 'd_angle')]
m_iml, m_iml_nolook = _iml(KEEP, 'iml'), _iml(KEEP_NOLOOK, 'iml_nolook')

METHODS = {'raw': m_raw, 'curve': m_curve, 'iml': m_iml, 'iml_nolook': m_iml_nolook}

## Run E1, E2a, E2b, E3

In [ ]:
def run(train_mask, test_mask, label):
    tr, te = D[train_mask], D[test_mask].reset_index(drop=True)
    FOLD['label'] = label + ' | test ' + ', '.join(sorted(te['person'].unique())[:2])
    out = te[['dataset', 'person', 'recording', 'vclass', 'exercise', 'y']].copy()
    for m, fn in METHODS.items():
        out[m] = fn(tr, te)
    out['experiment'] = label
    return out

def loso(subset, label):
    parts = []
    for p in sorted(D.loc[subset, 'person'].unique()):
        parts.append(run(subset & (D['person'] != p), D['person'] == p, label))
        print(f'  {label}: held out {p}', flush=True)
    return pd.concat(parts, ignore_index=True)

t0 = time.time()
is_fit, is_reh = D['dataset'] == 'fit3d', D['dataset'] == 'rehab24'
P = pd.concat([
    loso(is_fit, 'E1 Fit3D LOSO'),
    run(is_reh, is_fit, 'E2a REHAB->Fit3D'),
    run(is_fit, is_reh, 'E2b Fit3D->REHAB'),
    loso(is_fit | is_reh, 'E3 both LOSO'),
], ignore_index=True)
print(f'finished in {(time.time() - t0) / 60:.1f} min')

## Which hyperparameters did the inner loops choose?

In [ ]:
if TUNE:
    T = pd.DataFrame(TUNING_LOG)
    T['experiment'] = T['fold'].str.split(' | ', regex=False).str[0]
    T.to_csv(os.path.join(REPO, 'data', 'fit3d', 'tuning_log.csv'), index=False)
    for m, keys in (('curve', ['bin_width', 'min_count']),
                    ('iml', ['n_estimators', 'num_leaves', 'learning_rate', 'min_child_samples'])):
        print(f'{m}: how often each setting was chosen (columns = experiment)')
        display(T[T.method == m].groupby(keys + ['experiment']).size().unstack(fill_value=0))
    # a setting chosen in most folds is a stable, defensible choice for the live system

## Results

In [ ]:
M = list(METHODS)
err = P[M].sub(P['y'], axis=0).abs()
rec = pd.concat([P[['experiment', 'dataset', 'person', 'recording', 'vclass']], err], axis=1) \
        .groupby(['experiment', 'dataset', 'person', 'recording', 'vclass'], as_index=False)[M].mean()

def table(r):
    g = r.groupby('experiment')[M]
    t = g.median().round(2).add_suffix(' median')
    q = (g.quantile(.75) - g.quantile(.25)).round(2).add_suffix(' IQR')
    return pd.concat([t, q], axis=1).assign(recordings=g.size())

print('ALL VIEWS - per-recording MAE (deg)')
display(table(rec))
print('PROFILE VIEW ONLY - the live setup (user side-on to the camera)')
display(table(rec[rec.vclass == 'profile']))
print('BY VIEW CLASS, E3')
display(rec[rec.experiment == 'E3 both LOSO'].groupby('vclass')[M].median().round(2))

In [ ]:
# where in the range does each method fail? (curls need the 30-60 deg region)
P['band'] = pd.cut(P['y'], [0, 60, 90, 120, 150, 181], labels=['<60', '60-90', '90-120', '120-150', '>150'])
band = pd.concat([P[['experiment', 'band']], err], axis=1)
display(band[P['experiment'] == 'E3 both LOSO'].groupby('band', observed=True)[M].mean().round(2))

In [ ]:
# paired test per PERSON: is I_ML really better than the curve?
EMPTY = lambda label: dict(experiment=label, people=0, curve=np.nan, iml=np.nan,
                           gain_median=np.nan, iml_better_for='-', wilcoxon_p=np.nan)

def wil(a, b):
    try:
        return wilcoxon(a, b).pvalue
    except ValueError:                               # too few rows, or all differences zero
        return float('nan')

def paired(r, label):
    if not (r.experiment == label).any():
        return EMPTY(label)
    s = r[r.experiment == label].groupby('person')[['curve', 'iml']].mean()
    diff = s['curve'] - s['iml']                     # > 0 means I_ML is better
    p = wil(s['curve'], s['iml']) if len(s) >= 6 else float('nan')
    return dict(experiment=label, people=len(s), curve=round(s['curve'].median(), 2),
                iml=round(s['iml'].median(), 2), gain_median=round(diff.median(), 2),
                iml_better_for=f'{int((diff > 0).sum())}/{len(s)}', wilcoxon_p=round(p, 4))

def paired_rec(r, label):                           # E2: one model, many recordings
    s = r[r.experiment == label]
    if s.empty:
        return EMPTY(label)
    diff = s['curve'] - s['iml']
    return dict(experiment=label, people=s.person.nunique(), curve=round(s['curve'].median(), 2),
                iml=round(s['iml'].median(), 2), gain_median=round(diff.median(), 2),
                iml_better_for=f'{int((diff > 0).sum())}/{len(s)} rec.',
                wilcoxon_p=round(wil(s['curve'], s['iml']), 4))

for sub, name in ((rec, 'all views'), (rec[rec.vclass == 'profile'], 'profile only')):
    print(name)
    display(pd.DataFrame([paired(sub, 'E1 Fit3D LOSO'), paired_rec(sub, 'E2a REHAB->Fit3D'),
                          paired_rec(sub, 'E2b Fit3D->REHAB'), paired(sub, 'E3 both LOSO')]))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
t = rec[rec.vclass == 'profile'].groupby('experiment')[M].median()
t.plot.bar(ax=ax[0], rot=15); ax[0].set_ylabel('median per-recording MAE (deg)')
ax[0].set_title('Profile view (live setup)')
s = P[(P.experiment == 'E3 both LOSO') & (P.vclass == 'profile')]
s = s.sample(min(20000, len(s)), random_state=0)
for m, c in (('curve', 'tab:orange'), ('iml', 'tab:green')):
    ax[1].scatter(s['y'], s[m], s=1, alpha=.15, label=m, c=c)
ax[1].plot([0, 180], [0, 180], 'k--', lw=1); ax[1].set_xlabel('truth (deg)'); ax[1].set_ylabel('predicted (deg)')
ax[1].legend(markerscale=10); ax[1].set_title('E3, profile view')
plt.tight_layout(); plt.savefig(os.path.join(REPO, 'data', 'fit3d', 'curve_vs_iml.png'), dpi=150)

## Decision

Rule fixed **before** looking at the numbers (from the progress report):
adopt I_ML only if it is **≥ 4° better** than the curve on the **profile view**, the
improvement holds in **both cross-dataset directions**, and the per-person test in E3 is
significant (p < 0.05). Within **2°**, keep the curve. In between, keep the curve: it has
no 7-frame look-ahead (≈ 0.23 s), no model to ship, and is monotonic (it cannot invent a
non-physical angle).

In [ ]:
prof = rec[rec.vclass == 'profile']
e3 = paired(prof, 'E3 both LOSO')
e2a, e2b = paired_rec(prof, 'E2a REHAB->Fit3D'), paired_rec(prof, 'E2b Fit3D->REHAB')
gain = e3['gain_median']
avail = [e for e in (e2a, e2b) if np.isfinite(e['gain_median'])]
both_dirs = bool(avail) and all(e['gain_median'] > 0 for e in avail)
for e in (e2a, e2b):
    if not np.isfinite(e['gain_median']):
        print(f"{e['experiment']}: no profile test rows - not used in the decision")
print(f'E3 profile: curve {e3["curve"]} vs I_ML {e3["iml"]}  gain {gain:+.2f} deg  p={e3["wilcoxon_p"]}')
print(f'cross-dataset gain: REHAB->Fit3D {e2a["gain_median"]:+.2f}, Fit3D->REHAB {e2b["gain_median"]:+.2f}')
nl = prof[prof.experiment == 'E3 both LOSO'].groupby('person')[['iml', 'iml_nolook']].mean()
print(f'cost of removing the look-ahead (E3 profile): {(nl.iml_nolook - nl.iml).median():+.2f} deg')
if gain >= 4 and both_dirs and e3['wilcoxon_p'] < 0.05:
    CHOICE = 'iml'
    why = 'I_ML is >= 4 deg better, in both cross-dataset directions, and significant per person.'
elif gain <= 2:
    CHOICE = 'curve'
    why = 'I_ML is within 2 deg of the curve: not worth the look-ahead and the model.'
else:
    CHOICE = 'curve'
    why = ('I_ML is 2-4 deg better or not consistent across datasets: keep the curve, '
           'report I_ML as the better offline method.')
print('\nCHOICE:', CHOICE, '-', why)

## Fit the chosen method on all data (local only)

Anything fitted with Fit3D frames is derived from Fit3D, so it is written under
`data/fit3d/` (git-ignored). The public repo keeps the REHAB24-6-only curve.

In [ ]:
import datetime
out_dir = os.path.join(REPO, 'data', 'fit3d')
# final settings are tuned the same way: inner grouped CV over ALL people
FOLD['label'] = 'FINAL | all data'
hp = {}
if CHOICE == 'curve':
    hp = tune(D, curve_predict, GRID_CURVE, 'curve') if TUNE else CURVE_DEFAULT
    print('final curve settings:', hp)
    sub = D[D.vclass == 'profile']
    params = fit_one_curve(sub['angle_2d'], sub['y'], **hp)
    params.update(view_class='profile', date=str(datetime.date.today()),
                  source='REHAB24-6 + Fit3D profile frames (causal variant A)',
                  truth_range_5_95=[round(float(sub.y.quantile(.05)), 1), round(float(sub.y.quantile(.95)), 1)],
                  cv_profile_median_mae={'E3': e3['curve'], 'E2a': e2a['curve'], 'E2b': e2b['curve']})
    path = os.path.join(out_dir, 'population_curve_rehab_fit3d.json')
    save_params(path, params)
    print('saved', path)
    print('knots:', [(round(a), round(b)) for a, b in zip(params['knots_raw'], params['knots_corrected'])])
    print('use it locally:  python capture/calibrate_ui.py --side right --curve', path)
else:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        hp = tune(D, lambda a, b, **h: iml_predict(a, b, KEEP, **h), GRID_IML, 'iml') if TUNE else {}
        print('final I_ML settings:', hp)
        final = lgb.LGBMRegressor(**{**BASE, **hp, 'device_type': 'cpu'}).fit(D[KEEP], D['y'])
    path = os.path.join(out_dir, 'iml_rehab_fit3d.txt')
    final.booster_.save_model(path)
    json.dump({'features': KEEP, 'params': {**BASE, **hp}}, open(path + '.json', 'w'), indent=1)
    print('saved', path, '- next: wire it into demo_cv.py (7-frame look-ahead)')

In [ ]:
# numbers for the report
summary = {'all_views': table(rec).to_dict(), 'profile': table(prof).to_dict(),
           'paired_profile': [paired(prof, 'E1 Fit3D LOSO'), e2a, e2b, e3], 'choice': CHOICE, 'why': why, 'final_hyperparameters': hp}
with open(os.path.join(out_dir, 'curve_vs_iml_results.json'), 'w') as f:
    json.dump(summary, f, indent=1, default=str)
rec.to_csv(os.path.join(out_dir, 'curve_vs_iml_per_recording.csv'), index=False)
print('saved results json + per-recording csv')